# 03d — Exp 4: Decoupled Classifier Retraining (cRT)

**Project:** UREP 32-0210-250078 | Crack Classification

**Method:** Take the model from Exp 3 (CB focal loss). Freeze the entire backbone + CBAM.
Reset and retrain only the final dense layers for ~10 epochs with class-balanced sampling
(Kang et al., ICLR 2020 — cRT variant).

**Why:** Kang et al.'s key finding: representation learning prefers natural distribution;
only the classifier needs balanced data. Splits training into "learn good features" (Exp 3)
and "learn unbiased decisions" (this experiment). Negligible compute cost since backbone is frozen.

**Expected:** +1–3 macro-F1. Particularly helps shear recall.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import json
import numpy as np
import torch
import torch.nn as nn

import config
from src.device import print_device_summary, get_device, set_seed
from src.evaluation import evaluate_predictions
from src.model_cbam_hierarchical import (
    InceptionV3CBAMHierarchical, freeze_for_crt,
)
from src.hierarchical import (
    STAGE1_CLASSES, STAGE2_CLASSES, STAGE3_CLASSES,
    get_hierarchical_dataloaders,
    evaluate_hierarchical_model,
    hierarchical_pr_f1, per_stage_confusion_matrices, error_attribution,
)
from src.losses import train_hierarchical_model_v2

set_seed(config.RANDOM_SEED)

OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "exp4_crt")
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "logs"), exist_ok=True)

device_config = print_device_summary()
device = get_device()
STAGE_BATCH = device_config["batch_sizes"]
NUM_WORKERS = device_config["num_workers"]

print(f"\nExperiment 4: Decoupled Classifier Retraining (cRT)")
print(f"Device: {device}")

## Load Exp 3 model (CB focal loss trained)

In [ ]:
EXP3_PATH = os.path.join(config.OUTPUT_DIR, "exp3_cb_focal", "models", "best_model.pt")

model = InceptionV3CBAMHierarchical().to(device)
model.load_state_dict(torch.load(EXP3_PATH, map_location=device, weights_only=True))
print(f"Loaded Exp 3 model from {EXP3_PATH}")

## Freeze for cRT and reinitialize fc layers

In [ ]:
# Freeze backbone + CBAM, keep only fc layers trainable
freeze_for_crt(model)

# Reinitialize fc layer weights (Kaiming for Linear layers)
for head in [model.head1, model.head2, model.head3]:
    for m in head.fc.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)

print("Reinitialized all fc layers with Kaiming init.")

## Train fc layers with class-balanced sampling

Joint sampler (25% no_crack mix), all 3 heads active, standard masked CE (not focal —
Kang et al. showed cRT works best with standard CE + balanced sampling).

Only 10 epochs needed since backbone features are frozen and high-quality.

In [ ]:
CRT_EPOCHS = 10
CRT_LR = 1e-4

train_loader, val_loader, test_loader = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[1],   # frozen backbone = low memory
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="joint",
    no_crack_mix=0.25,
)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=CRT_LR,
)

history = train_hierarchical_model_v2(
    model, train_loader, val_loader, optimizer, device,
    epochs=CRT_EPOCHS, output_dir=OUTPUT_DIR, stage=1,
    loss_weights=(0.2, 0.3, 0.5),
    loss_fn_per_stage=None,  # standard masked CE
    patience=CRT_EPOCHS,     # no early stopping for cRT
    model_name="exp4_crt",
)

## Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
metric_pairs = [
    ("loss_s1", "val_loss_s1", "Stage 1 loss"),
    ("loss_s2", "val_loss_s2", "Stage 2 loss"),
    ("loss_s3", "val_loss_s3", "Stage 3 loss"),
    ("acc_s1",  "val_acc_s1",  "Stage 1 accuracy"),
    ("acc_s2",  "val_acc_s2",  "Stage 2 accuracy"),
    ("acc_s3",  "val_acc_s3",  "Stage 3 accuracy"),
]

for ax, (tk, vk, title) in zip(axes.flat, metric_pairs):
    n = len(history[tk])
    xs = range(n)
    ax.plot(xs, history[tk], "b-", label="train")
    ax.plot(xs, history[vk], "b--", label="val")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.suptitle("Exp 4: cRT (fc-only retraining)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "training_history_exp4.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## Save model

In [ ]:
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "models", "best_model.pt"))
print("Saved cRT model.")

## Test-set evaluation

In [ ]:
_, _, test_loader = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[1],
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="stage1",
)

results = evaluate_hierarchical_model(model, test_loader, device, t1=0.5, t2=0.5)

y_true_idx = np.array([config.CLASS_NAMES.index(c) for c in results["y_true_flat"]])
y_pred_idx = np.array([config.CLASS_NAMES.index(c) for c in results["y_pred_flat"]])
metrics = evaluate_predictions(
    y_true_idx, y_pred_idx,
    output_dir=OUTPUT_DIR, model_name="exp4_crt",
)

## Per-stage confusion matrices, hierarchical metrics, error attribution

In [ ]:
import seaborn as sns

stage_cms = per_stage_confusion_matrices(results["y_true_paths"], results["y_pred_paths"])
h_metrics = hierarchical_pr_f1(results["y_true_paths"], results["y_pred_paths"])
err_attr  = error_attribution(results["y_true_paths"], results["y_pred_paths"])

print("Hierarchical precision/recall/F1:")
for k, v in h_metrics.items():
    print(f"  {k}: {v:.4f}")

print(f"\nError attribution:")
print(f"  Total:        {err_attr['total']}")
print(f"  Correct:      {err_attr['correct']}")
print(f"  Stage1 errs:  {err_attr['errors_by_stage']['stage1']}")
print(f"  Stage2 errs:  {err_attr['errors_by_stage']['stage2']}")
print(f"  Stage3 errs:  {err_attr['errors_by_stage']['stage3']}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key in zip(axes, ["stage1", "stage2", "stage3"]):
    if key not in stage_cms:
        ax.set_visible(False); continue
    info = stage_cms[key]
    sns.heatmap(info["cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=info["classes"], yticklabels=info["classes"], ax=ax)
    ax.set_title(f"{key}  ({info['cm'].sum()} samples)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "per_stage_confusion_matrices.png"),
            dpi=150, bbox_inches="tight")
plt.show()

with open(os.path.join(OUTPUT_DIR, "exp4_results.json"), "w") as f:
    json.dump({
        "crt_epochs": CRT_EPOCHS,
        "crt_lr": CRT_LR,
        "hierarchical": h_metrics,
        "error_attribution": err_attr,
        "flat": {
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
        },
    }, f, indent=2)
print(f"\nSaved results to {OUTPUT_DIR}/exp4_results.json")